# N034 · 差分与批量区间更新

**目标：** 把重复区间更新转化为边界事件。

**先修：** N031, N033。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 一维/二维差分；前缀还原；端点抵消；闭区间转半开。

**配套讲解来源：** [同名逐章意见](../../comment/034_difference_arrays.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决**批量区间更新**问题：长度为 n 的数组一开始全是 0，然后来一批更新指令，每条指令说"把下标 l 到 r−1 这一段每个位置都加上 value"。全部执行完后，输出最终数组。

具体例子（“运行示例”部分/“自动测试”部分 的数据）：n=5，三条更新 `[(0,2,10),(1,3,20),(1,5,25)]`（含义分别是 [0,2) 加 10、[1,3) 加 20、[1,5) 加 25），最终输出 `[10,55,45,25,25]`。我们手验下标 1：它被 [0,2) 的 10 和 [1,3) 的 20 和 [1,5) 的 25 同时覆盖，10+20+25=55；下标 0 只被第一条覆盖，是 10；下标 3 被 [1,5) 的 25 覆盖（[1,3) 不含 3），是 25。对上了。

航班版的坐标换算也提前演示一下：一基闭区间 [first,last] 说"给第 first 到第 last 个位置加值"。第 first 个位置对应零基下标 first−1，第 last 个对应 last−1，半开右端再进一格就是 last。所以换算公式是零基半开 **[first−1, last)**。拿"给第 1 到第 2 个位置各加 10"演练：[1,2] 闭 → 零基下标 0、1 → 半开 [0,2)，正好是本节例子里的第一条更新 (0,2,10)。

如果每条指令都老老实实循环 l..r−1 加一遍，q 条指令最坏要做 n×q 次加法。差分的妙处是把每条指令从 O(区间长度) 降到 O(1)：只在两个边界上记账，最后扫一遍还原。

本章还带一个同思想的应用题**拼车**：给定一批行程 `[乘客数, 上车站, 下车站]`和车容量，问旅途中任意时刻车上人数是否都不超载。例子：行程 `[[2,1,5],[3,3,7]]`、容量 4，答案是 `False`——第二拨 3 人在站 3 上车时，车上已有 2 人，共 5 人超过 4。拼车本质也是"区间加"：每条行程给 [start,end) 这段路的车载量加一个常数，问峰值是否越界。

## 2. 基础知识：先读懂这些词

- **差分数组（difference array）**：diff[i] 描述"相邻位置的增量变化"。直观理解：diff[i] = 最终数组 a[i] − a[i−1]（a[−1] 视为 0）。反过来，把 diff 从左到右做前缀和累加，就能还原出 a。差分和前缀和是一对互逆运算：**差分的路是前缀和，前缀和的路是差分**。
  - 一个帮助记忆的说法：前缀和把"点上的账"汇总成"到这里的总量"，差分把"一段的总量"拆回"点上的账"。往返一趟信息不丢，代价都是一次线性扫。
- **边界事件（端点记账）**：一条"区间 [l,r) 整体加 v"的指令，对差分数组的影响只有两处：`diff[l]+=v`（从 l 开始有一股新的增量启动）和 `diff[r]-=v`（到 r 这股增量结束）。中间的位置差分不变，因为它们和左邻居一起被加了同样的数。每条指令 O(1) 就记完了。
  - 生活比喻：小区门口发临时通行证，进门的登记"+1 人"，出门登记"−1 人"，小区里此刻有多少人只需把登记簿从早到晚累加——你不需要数小区里每一个人。
- **前缀还原**：所有指令记完账后，从左到右 `total+=diff[i]`，total 就是位置 i 的总增量。这一步等价于把所有"启动/结束"事件叠加：total 在 l 处抬升 v，在 r 处跌回，所以区间内恰好是 v、区间外归零。
- **闭区间转半开**：本章更新接口统一用零基半开 [l,r)。像航班预订题给的是一基闭区间 [first,last]，转换公式是 l=first−1、r=last（闭区间的 last 是最后一个元素，半开的 r 是它加一）。
- **稀疏事件表（拼车版）**：拼车问题里站号可能很大（0~1000），但行程条数不多。与其开大数组，不如用字典 `events` 把"某站上车几人、下车几人"累计起来，再按站号排序扫描。同站既有上车又有下车时用 `events[start]+p` 和 `events[end]-p` 分别记账，字典会自动把同站的事件合并。
  - 什么时候选数组差分、什么时候选事件表？看"位置范围 n"和"指令条数 q"的相对大小：q 条指令最多产生 2q 个事件，若 n 远大于 2q（比如站号到 10^9 但只有几百条行程），事件表省时又省内存；若位置密集、更新铺满全区间，数组差分更直接。
- **末端哨兵（diff 长度 n+1）**：代码里 `diff=[0]*(n+1)`，多开一格。因为 `diff[r]-=v` 在 r=n（更新延伸到数组末尾）时也要有地方写，这多出来的一格就像跑道尽头的缓冲垫，专门收容"结束标记"，扫描时只扫前 n 格，哨兵不会被输出。

## 3. 思路推导：从小例子开始

- **Step 1**：开长度 n+1 的全零差分数组 diff。
- **Step 2**：对每条更新 (left,right,value)（先检查 0≤left≤right≤n 合法），只做两次加法：`diff[left]+=value`、`diff[right]-=value`。
- **Step 3**：所有指令记完后，从 i=0 扫到 n−1，维护滚动和 `total+=diff[i]`，total 依次就是每个位置的最终增量，收集成答案数组。

手算演示（n=5，`updates=[(0,2,10),(1,3,20),(1,5,25)]`，“运行示例”部分 的数据）：

| 位置 | diff（差分事件） | 重建值（total 累加） |
|---|---|---|
| 0 | +10 | 10 |
| 1 | +20 +25 = +45 | 55 |
| 2 | −10 | 45 |
| 3 | −20 | 25 |
| 4 | 0 | 25 |
| 5 | −25 | （边界哨兵，不输出） |

逐条看记账过程：指令 (0,2,10) 把 diff 变成 [10,0,−10,0,0,0]；指令 (1,3,20) 变成 [10,20,−10,−20,0,0]；指令 (1,5,25) 变成 [10,45,−10,−20,0,−25]。从左累加：10→55→45→25→25，正是答案 `[10,55,45,25,25]`。这张表就是 notebook “运行示例”部分 用 `show_table` 打印的内容（列：位置、差分事件、重建值，最后一行标着"边界哨兵"）。

拼车问题的推导：

- **Step 1**：把每条行程 [passengers,start,end] 拆成两个事件：start 站 +passengers，end 站 −passengers。同站的多个事件在字典里累加。
  - 例：两条行程 [[2,1,5],[3,5,7]] 在站 5 各有一笔 −2 和 +3，合并成 events[5]=+1，扫描到站 5 时一次性结算"前两人下车、后三人上车"。
- **Step 2**：按站号从小到大扫描事件，维护车上人数 load。这里隐藏着半开语义：乘客在 end 站已经下车，不再占座，所以 [start,end) 是半开占用。
- **Step 3**：任一时刻 load 超过容量立刻返回 False；扫完没超就返回 True。

手算演示（`trips=[[2,1,5],[3,3,7]]`，capacity=4）：events = {1:+2, 5:−2, 3:+3, 7:−3}。排序后逐站结算：

| 站号 | 事件 | 结算后 load | 是否超载（容量 4） |
|---|---|---|---|
| 1 | +2 | 2 | 否 |
| 3 | +3 | 5 | **是，返回 False** |
| 5 | −2 | （未执行到） | — |
| 7 | −3 | （未执行到） | — |

对比 `trips=[[2,1,5],[3,5,7]]`、capacity=3：events={1:+2, 5:(−2+3)=+1, 7:−3}。站 1 后 load=2；站 5 时第一拨下车与第二拨上车在同一 key 里合并结算，load=2+1=3 ≤ 3；站 7 后归 0，返回 True。这两个例子合起来说明：超载只可能发生在"某站结算完"的瞬间，逐站检查就够。

## 4. 核心代码：apply_range_add

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def apply_range_add(n, updates):
    diff = [0] * (n + 1)
    for left, right, value in updates:
        if not 0 <= left <= right <= n:
            raise IndexError('invalid range')
        diff[left] += value
        diff[right] -= value
    answer = []
    total = 0
    for i in range(n):
        total += diff[i]
        answer.append(total)
    return answer
```

### 逐段读懂代码

先看 `apply_range_add`，对应实现如下：

```python
def apply_range_add(n, updates):
    diff = [0] * (n + 1)
    for left, right, value in updates:
        if not 0 <= left <= right <= n:
            raise IndexError('invalid range')
        diff[left] += value
        diff[right] -= value
    answer = []
    total = 0
    for i in range(n):
        total += diff[i]
        answer.append(total)
    return answer
```

- `diff=[0]*(n+1)`：长度 n+1 的差分表，多出的最后一格是末端哨兵，供 `diff[right]` 在 right=n 时落笔。
- `for left,right,value in updates:`：直接把三元组解包成三个变量，比 `upd[0]`、`upd[1]` 可读。
- `if not 0<=left<=right<=n: raise IndexError('invalid range')`：链式比较一次检查四个条件（left≥0、left≤right、right≤n），非法区间（如 left>right 或越界）直接抛异常，宁可崩溃也不给错答案。注意上界写的是 n 而不是 n−1：半开区间的右端允许等于 n（表示"加到最后一个位置"），diff 长度 n+1 里的哨兵格正是为此准备的。
- `diff[left]+=value; diff[right]-=value`：一条区间更新压缩成两笔账——起点启动增量，结束点（半开，即最后一个被加位置的下一格）抵消增量。分号把两件密切相关的事写在一行。
  - 特别提醒 `diff[right]` 不是 `diff[right-1]`：半开约定里 right 本身**不在**被更新范围内，抵消记号要写在"范围外第一格"，前缀还原时这一格才会把增量准确跌回。
  - 拿 (0,2,10) 验证：diff[0]+=10、diff[2]-=10。还原 total 在 i=0、i=1 时是 10，i=2 时 10−10=0——恰好 [0,2) 加 10、其他位置不加。
- `answer=[]; total=0`：answer 收结果，total 是滚动前缀和。
- `for i in range(n): total+=diff[i]; answer.append(total)`：只扫前 n 格做前缀还原；total 加上当前差分后恰好等于位置 i 的总增量，append 进答案。哨兵格（下标 n）不参与输出。

再看 `car_pooling`，对应实现如下：

```python
def car_pooling(trips, capacity):
    events = {}
    for passengers, start, end in trips:
        events[start] = events.get(start, 0) + passengers
        events[end] = events.get(end, 0) - passengers
    load = 0
    for position in sorted(events):
        load += events[position]
        if load > capacity:
            return False
    return True
```

- `events={}`：稀疏事件表，key 是站号，value 是该站净上车人数（负数表示下车）。
- `events[start]=events.get(start,0)+passengers`：`dict.get(key,0)` 表示"key 不存在就当 0"，这样同站多笔事件自然累加，不用预判 key 在不在。
  - 一条行程拆成上行、下行两笔账，本质就是一维差分"起点加、终点减"的字典版：乘客载重曲线在 [start,end) 上是一个高度为 passengers 的平台。
- `events[end]=events.get(end,0)-passengers`：end 站记负数。半开语义：乘客在 end 站不算在车上。
- `load=0`：车上人数从 0 开始。
- `for position in sorted(events):`：按站号从小到大扫事件。字典本身无序，必须排序；事件数最多是行程数的两倍，排序很快。
- `load+=events[position]; if load>capacity: return False`：把本站所有上下车一次性结算进 load，任何一站结算完超载就判 False。注意同一站先上后下（或同站有上有下）都在同一 key 里合并了，不会因为顺序歧义误判。
- `return True`：全程未超载。

## 5. 分段实现：按顺序运行

**本章接口：** `apply_range_add(n, updates)`、`car_pooling(trips, capacity)`

### apply_range_add

In [1]:
def apply_range_add(n, updates):
    diff = [0] * (n + 1)
    for left, right, value in updates:
        if not 0 <= left <= right <= n:
            raise IndexError('invalid range')
        diff[left] += value
        diff[right] -= value
    answer = []
    total = 0
    for i in range(n):
        total += diff[i]
        answer.append(total)
    return answer

### car_pooling

In [2]:
def car_pooling(trips, capacity):
    events = {}
    for passengers, start, end in trips:
        events[start] = events.get(start, 0) + passengers
        events[end] = events.get(end, 0) - passengers
    load = 0
    for position in sorted(events):
        load += events[position]
        if load > capacity:
            return False
    return True

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

updates=[(0,2,10),(1,3,20),(1,5,25)]; diff=[0]*6
for l,r,v in updates: diff[l]+=v; diff[r]-=v
out=apply_range_add(5,updates)
show_table(['位置','差分事件','重建值'],[(i,diff[i],out[i] if i<5 else '边界哨兵') for i in range(6)])

位置,差分事件,重建值
0,10,10
1,45,55
2,-10,45
3,-20,25
4,0,25
5,-25,边界哨兵


## 7. 正确性、适用条件与复杂度

每个更新的前缀累积在l前为0、[l,r)为value、r后归零；线性叠加得到全部更新。拼车同站下车与上车可以同时汇总，区间按半开占用处理。

**代价：** 差分 O(n+q) 时间、O(n) 空间；稀疏拼车事件排序 O(q log q)、O(q) 空间。

### 展开理解

**为什么对（差分）**：单独看一条更新 (l,r,v)。它给 diff[l] 加了 v、给 diff[r] 减了 v。做前缀还原时，total 在扫过 l 之前不含这 v（那几格 diff 没动），从 l 到 r−1 每格都含这 v（v 已累进 total 且尚未被减回），扫过 r 之后 v 又被减掉（diff[r] 的 −v 累进 total 抵消了）。也就是说，单条更新还原出来的增量曲线在 [l,r) 恰好是 v、区间外是 0。多条更新互不干扰，每条都贡献这样一段平台，加起来就是总增量——这就是 “正确性与复杂度”部分 说的"每个更新的前缀累积在 l 前为 0、[l,r) 为 value、r 后归零；线性叠加得到全部更新"。

**为什么对（拼车）**：每个乘客的载重曲线同样是"start 到 end−1 的平台"，同站上下车合并进同一 key 后按时间序结算，load 在每个站号处的值就是该时刻车上真实人数。超载与否只在上下车站点发生变化，所以只查站点就够，不需要逐公里检查。

**复杂度**：差分版记 q 条账是 O(q)，还原扫 n 格是 O(n)，共 O(n+q)；空间 O(n)。具体数字：n=10^5、q=10^5 时约二十万次操作，毫秒级；若逐条指令暴力加区间，最坏 10^5×10^5=百亿次加法，完全跑不动。拼车版不依赖站号范围，只对 2q 个事件排序，O(q log q)；q=10^4 时排序约 10^4×14≈14 万次比较，同样瞬间完成；空间 O(q)。这也解释了为什么站号范围大、行程少时用稀疏事件表更划算。

差分与前缀和的关系值得再强调一遍：它们是同一枚硬币的两面。N031/N033 里我们"先做前缀和，之后 O(1) 查询区间和"；本章反过来"先记差分，O(1) 完成区间加，最后扫一遍前缀还原"。什么时候用哪一面，取决于操作模式——查询多、数组不变就用前缀和；更新多、最后一次性看结果就用差分。两种场景都是用一次线性扫，把反复的区间工作摊平。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条解释：

- `bookings=[[1,2,10],[2,3,20],[2,5,25]]` 加 `assert apply_range_add(5,[(a-1,b,v) for a,b,v in bookings])==[10,55,45,25,25]`：正常值测试，同时测**坐标转换**。这里模拟航班预订题的一基闭区间输入：第 1~2 号座位（一基闭 [1,2]）就是零基半开 [0,2)，转换 `a-1` 与 `b` 正是 “思路推导”部分 说的 `[first−1, last)`。逐条转换：[1,2,10]→(0,2,10)、[2,3,20]→(1,3,20)、[2,5,25]→(1,5,25)，恰好就是第三节手算的那组更新，期望值 [10,55,45,25,25] 也一字不差。
- `assert car_pooling([[2,1,5],[3,3,7]],4) is False`：正常值测试。站 3 时 2+3=5 > 4，必须判超载，考察"超载只在事件站点被检出"的结算逻辑。
- `assert car_pooling([[2,1,5],[3,5,7]],3) is True`：边界值测试。第一拨在站 5 下车、第二拨恰在站 5 上车，半开语义下两者同时结算：load=2−2+3=3 ≤ 3，不超。这条专门卡"下车算晚一站"的 off-by-one——如果 end 站还占座，这里会误判 2+3=5>3 而错答 False。
- 随机段：`rng=Random(34)` 固定种子可复现。对 n=1..9 各生成 30 条随机更新，参考实现 `ref` 逐条暴力对区间内每格加 v，最后对拍。几个细节：
  - `l=rng.randrange(n+1); r=rng.randrange(l,n+1)`：先取左端再在 [l,n] 内取右端，天然保证 l≤r 合法；l==r 的空更新（不改变任何格子）也会被抽到。
  - `v=rng.randrange(-5,6)`：值域含负数和零，验证差分对"减法更新"同样成立——公式里 value 只是系数，正负通吃。
  - `for i in range(l,r): ref[i]+=v`：参考实现就是差分要取代的那个 O(区间长度) 暴力，30 条更新 × 最多 9 格的朴素累加，用它当裁判完全够用。

In [4]:
bookings = [[1, 2, 10], [2, 3, 20], [2, 5, 25]]

assert apply_range_add(5, [(a - 1, b, v) for a, b, v in bookings]) == [10, 55, 45, 25, 25]

assert car_pooling([[2, 1, 5], [3, 3, 7]], 4) is False

assert car_pooling([[2, 1, 5], [3, 5, 7]], 3) is True

from random import Random

rng = Random(34)

for n in range(1, 10):
    updates = []
    ref = [0] * n
    for _ in range(30):
        l = rng.randrange(n + 1)
        r = rng.randrange(l, n + 1)
        v = rng.randrange(-5, 6)
        updates.append((l, r, v))
        for i in range(l, r):
            ref[i] += v
    assert apply_range_add(n, updates) == ref

print('N034: 所有本章断言通过')

N034: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 实现矩形加值的二维差分。

**练习 2：** 解释末端哨兵单元。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（实现矩形加值的二维差分）**：提示：把"一维区间的两个端点"升级为"二维矩形的四个角"。对矩形 [r1,r2)×[c1,c2) 加 v，在差分表上记四笔：`d[r1][c1]+=v`、`d[r1][c2]-=v`、`d[r2][c1]-=v`、`d[r2][c2]+=v`——加 v 的角启动增量，减 v 的角在行或列方向抵消，右下角被两个方向的抵消多减了一次所以加回。还原时对表做二维前缀和（复用 N033 的容斥公式）。验证可用 2×3 小矩阵、一条更新 (0,1,2,3,+5) 手算：期望矩形内加 5、矩形外不变，再和暴力对拍。
- **练习 2（解释末端哨兵单元）**：提示：想一条覆盖到数组末尾的更新，比如 n=5、指令 (2,5,7)。`diff[5]-=7` 这笔账写到哪？如果 diff 只开 n=5 格，下标 5 越界。多开的一格让"结束标记"永远有地方放；还原循环 `for i in range(n)` 只读前 5 格，哨兵里的 −7 只用于让"假想的“运行示例”部分"归零，本身不输出。你可以在纸上画 diff 全表（含哨兵），再写一个 n=3、指令 (0,3,4) 的例子演示哨兵被写、但不被读的过程。
  - 对应表格：diff = [4,0,0,−4]，还原 total 依次 4、4、4，答案 [4,4,4]；哨兵 diff[3]=−4 从未被 range(3) 读到，但少了它第 (0,3,4) 条指令就无处注销。
  - 另一种等价写法是特判 `if right<n: diff[right]-=value`，用分支换一格内存——两种写法都对，本章选哨兵是因为它让循环体保持无分支。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def apply_range_add(n, updates):
    diff = [0] * (n + 1)
    for left, right, value in updates:
        if not 0 <= left <= right <= n:
            raise IndexError('invalid range')
        diff[left] += value
        diff[right] -= value
    answer = []
    total = 0
    for i in range(n):
        total += diff[i]
        answer.append(total)
    return answer

def car_pooling(trips, capacity):
    events = {}
    for passengers, start, end in trips:
        events[start] = events.get(start, 0) + passengers
        events[end] = events.get(end, 0) - passengers
    load = 0
    for position in sorted(events):
        load += events[position]
        if load > capacity:
            return False
    return True

# 示例与回归测试
bookings = [[1, 2, 10], [2, 3, 20], [2, 5, 25]]

assert apply_range_add(5, [(a - 1, b, v) for a, b, v in bookings]) == [10, 55, 45, 25, 25]

assert car_pooling([[2, 1, 5], [3, 3, 7]], 4) is False

assert car_pooling([[2, 1, 5], [3, 5, 7]], 3) is True

from random import Random

rng = Random(34)

for n in range(1, 10):
    updates = []
    ref = [0] * n
    for _ in range(30):
        l = rng.randrange(n + 1)
        r = rng.randrange(l, n + 1)
        v = rng.randrange(-5, 6)
        updates.append((l, r, v))
        for i in range(l, r):
            ref[i] += v
    assert apply_range_add(n, updates) == ref

print('N034: 所有本章断言通过')

N034: 所有本章断言通过


## 11. 代表题与迁移

- **1109. Corporate Flight Bookings**：航班预订（每条预订给一段连续航班座位数加上 bookings 值），练的就是 `apply_range_add` 加上一基闭区间到零基半开的坐标转换。这题如果对每条预订暴力累加，n 和 q 都是两万量级时最坏四亿次加法会很吃力，差分 O(n+q) 秒出。
- **1094. Car Pooling**：拼车判超载，练的是 `car_pooling` 的稀疏事件表——"区间加"思想在不宜开大数组时的字典化变体，外加半开占用语义。注意该题乘客在 end 站可立即被下一批接上（同站换客不叠加），半开事件合并恰好表达了这个语义。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。